# Technical Crypto Modeling

Notebook dedicado ? avalia??o dos modelos por horizonte e an?lise de import?ncia das features t?cnicas.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data.historical_loader import load_historical_asset_dataframe
from src.data.technical_preprocessing import prepare_technical_market_dataframe
from src.features.technical_indicators import (
    DEFAULT_TECHNICAL_FEATURES,
    REDUCED_TECHNICAL_FEATURES,
    DEFAULT_TECHNICAL_HORIZONS,
    add_technical_indicators,
    add_technical_targets,
)
from src.models.blockchain_training import build_default_models
from src.models.technical_training import evaluate_feature_importance_by_horizon, evaluate_technical_models_by_horizon


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'CSvs'
ASSET_NAME = 'Binance_BTCUSDT_d'
HORIZONS = DEFAULT_TECHNICAL_HORIZONS
FULL_FEATURES = list(DEFAULT_TECHNICAL_FEATURES)
REDUCED_FEATURES = list(REDUCED_TECHNICAL_FEATURES)
MODELS = build_default_models()


In [ ]:
raw_asset = load_historical_asset_dataframe(DATA_DIR, ASSET_NAME)
technical_base = prepare_technical_market_dataframe(raw_asset)
technical_df = add_technical_indicators(technical_base)
technical_df = add_technical_targets(technical_df, horizons=HORIZONS)
technical_df = technical_df.dropna()


In [ ]:
baseline_results, baseline_artifacts = evaluate_technical_models_by_horizon(
    technical_df,
    FULL_FEATURES,
    HORIZONS,
    models=MODELS,
)
baseline_results


In [ ]:
importance_results, feature_importance_dict, importance_artifacts = evaluate_feature_importance_by_horizon(
    technical_df,
    REDUCED_FEATURES,
    HORIZONS,
    models=MODELS,
)
importance_results


In [ ]:
for model_name, horizons in feature_importance_dict.items():
    importance_df = pd.DataFrame(horizons, index=REDUCED_FEATURES)
    plt.figure(figsize=(15, 8))
    sns.heatmap(importance_df, cmap='YlOrRd', annot=True, fmt='.3f')
    plt.title(f'{model_name} - Feature Importance Across Horizons')
    plt.xlabel('Horizon')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
